# Training of the MIL Gated Attention regressor

In [1]:
# imports
import os
import pandas as pd
from tqdm import tqdm
if os.getcwd().endswith("notebooks"):
    os.chdir("../")

from src.models.attention_mil import run_mil_training
import torch


In [2]:
video_tcn_embeddings_train = torch.load("data/embeddings/video_tcn_embeddings_train.pt")
video_tcn_embeddings_val = torch.load("data/embeddings/video_tcn_embeddings_val.pt")

/var/folders/sj/219f9qbn3y15yynshcyrrp3c0000gn/T/ipykernel_37896/3678603084.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  video_tcn_embeddings_train = torch.load("data

In [11]:
video_tcn_embeddings_train[0]

(tensor([[ 1.5004,  1.1677, -1.6129,  ..., -1.0435,  1.4814,  1.5214],
         [ 1.2951,  1.0056, -1.6177,  ..., -1.3734,  0.9658,  1.6164],
         [ 1.3075,  0.9537, -1.6059,  ..., -1.5193,  1.0556,  1.5899],
         ...,
         [ 1.2432,  0.8980, -1.6198,  ..., -1.1858,  1.5179,  1.5943],
         [ 1.2766,  1.1121, -1.5334,  ..., -1.2042,  1.7761,  1.4690],
         [ 1.3488,  0.6573, -1.3455,  ..., -1.3755,  2.1343,  1.9632]]),
 tensor([40.5000]),
 ('2024-01-15_17-57-25', 4))

In [3]:
test = video_tcn_embeddings_train[0][0]

In [3]:
final_model, best_corr = run_mil_training(video_tcn_embeddings_train, video_tcn_embeddings_val, num_epochs=300)

Starting MIL Training on CPU for 300 epochs...
Training Bags: 65 | Validation Bags: 18
Epoch 01 | Train Loss: 270.6069 | Val Loss: 171.6426 | Val Corr: 0.5550
>>> Saved new best model checkpoint.
Epoch 02 | Train Loss: 214.2209 | Val Loss: 126.8681 | Val Corr: 0.6346
>>> Saved new best model checkpoint.
Epoch 03 | Train Loss: 170.8894 | Val Loss: 93.4723 | Val Corr: 0.7514
>>> Saved new best model checkpoint.
Epoch 04 | Train Loss: 137.3379 | Val Loss: 71.8627 | Val Corr: 0.8000
>>> Saved new best model checkpoint.
Epoch 05 | Train Loss: 115.4979 | Val Loss: 58.0143 | Val Corr: 0.7969
Epoch 06 | Train Loss: 102.1930 | Val Loss: 50.4197 | Val Corr: 0.7979
Epoch 07 | Train Loss: 93.5358 | Val Loss: 47.6783 | Val Corr: 0.7959
Epoch 08 | Train Loss: 88.7184 | Val Loss: 46.0280 | Val Corr: 0.7948
Epoch 09 | Train Loss: 85.7265 | Val Loss: 45.4459 | Val Corr: 0.7948
Epoch 10 | Train Loss: 83.8847 | Val Loss: 45.4825 | Val Corr: 0.7948
Epoch 11 | Train Loss: 82.5151 | Val Loss: 45.4271 | Val 

KeyboardInterrupt: 

# Add Hand Crafted Metric(s) => Hybrid Model

In [4]:
from src.models.hybrid_attention_mil import run_mil_training, prepare_hybrid_mil_bags

In [13]:
df_metrics.head()

,Vid_Name,total_path_Right,total_duration_Left,num_reversals_Right
0,2024-01-15_13-18-23,46161.735043,537.133333,730
1,2024-01-15_13-37-36,26742.389283,287.833333,370
2,2024-01-15_14-03-23,44587.971892,422.933333,697
3,2024-01-15_14-32-45,81538.611520,1282.433333,1570
4,2024-01-15_15-05-31,64704.176039,1011.033333,1129


In [5]:
df_metrics = pd.read_csv("data/metrics/right_total_metrics.csv")

# 1. Define the column name you want to use for the fusion
TPL_METRIC_NAME = 'total_path_Right' 
mean = df_metrics[TPL_METRIC_NAME].mean()
std = df_metrics[TPL_METRIC_NAME].std()

# 2. Create the new, correctly formatted lists
hybrid_video_embeddings_train = prepare_hybrid_mil_bags(
    embeddings_list=video_tcn_embeddings_train, 
    metrics_df=df_metrics, 
    metric_column_name=TPL_METRIC_NAME,
    mean=mean,
    std=std
)

hybrid_video_embeddings_val = prepare_hybrid_mil_bags(
    embeddings_list=video_tcn_embeddings_val, 
    metrics_df=df_metrics, 
    metric_column_name=TPL_METRIC_NAME,
    mean=mean,
    std=std
)

In [6]:
_, _ = run_mil_training(hybrid_video_embeddings_train, hybrid_video_embeddings_val, num_epochs=500, lr = 3e-5
)

Starting HYBRID MIL Training on CPU for 500 epochs...
Training Bags: 65 | Validation Bags: 18
Epoch 01 | Train Loss: 2231.5876 | Val Loss (MSE): 2069.3847 | Val RMSE: 45.4905 | Val MAE: 44.9215 | Val Corr: 0.6284
Epoch 02 | Train Loss: 2210.7370 | Val Loss (MSE): 2048.7370 | Val RMSE: 45.2630 | Val MAE: 44.6915 | Val Corr: 0.6398
Epoch 03 | Train Loss: 2173.5858 | Val Loss (MSE): 2027.7028 | Val RMSE: 45.0300 | Val MAE: 44.4561 | Val Corr: 0.6222
Epoch 04 | Train Loss: 2171.2902 | Val Loss (MSE): 2007.7792 | Val RMSE: 44.8082 | Val MAE: 44.2316 | Val Corr: 0.5985
Epoch 05 | Train Loss: 2171.5379 | Val Loss (MSE): 1988.3902 | Val RMSE: 44.5914 | Val MAE: 44.0121 | Val Corr: 0.5964
Epoch 06 | Train Loss: 2103.2633 | Val Loss (MSE): 1968.0742 | Val RMSE: 44.3630 | Val MAE: 43.7813 | Val Corr: 0.6160
Epoch 07 | Train Loss: 2099.6991 | Val Loss (MSE): 1948.8339 | Val RMSE: 44.1456 | Val MAE: 43.5614 | Val Corr: 0.5974
Epoch 08 | Train Loss: 2076.4202 | Val Loss (MSE): 1929.1826 | Val RMSE: 

# Try LOSOCV

In [8]:
import torch
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split


In [9]:
# Load full dataset
video_tcn_embeddings = torch.load("data/embeddings/video_tcn_embeddings.pt")

# Load metrics
df_metrics = pd.read_csv("data/metrics/right_total_metrics.csv")

# Metric column
TPL_METRIC_NAME = 'total_path_Right'

# Normalize using FULL dataset statistics (correct for LOSO)
mean = df_metrics[TPL_METRIC_NAME].mean()
std = df_metrics[TPL_METRIC_NAME].std()


/var/folders/sj/219f9qbn3y15yynshcyrrp3c0000gn/T/ipykernel_37160/2282783242.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  video_tcn_embeddings = torch.load("data/embed

In [10]:
surgeon_ids = sorted(list(set([x[2][1] for x in video_tcn_embeddings])))
print("Surgeons found:", surgeon_ids)


Surgeons found: [1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]


In [11]:
def run_loso_hybrid_mil(
    all_embeddings,
    metrics_df,
    metric_column_name,
    mean,
    std,
    num_epochs=300,
    lr=1e-4
):
    """
    Leave-One-Surgeon-Out Cross Validation for Hybrid MIL
    """

    # Extract unique surgeons
    surgeons = sorted(list(set([x[2][1] for x in all_embeddings])))

    results = defaultdict(dict)
    fold = 1

    print("\n======= STARTING LOSO CROSS VALIDATION =======")
    print(f"Total surgeons: {len(surgeons)}")

    for test_surgeon in surgeons:
        print(f"\n==============================")
        print(f"FOLD {fold}/{len(surgeons)} — TEST SURGEON: {test_surgeon}")
        print(f"==============================")

        # -----------------------------
        # SPLIT BY SURGEON
        # -----------------------------
        train_data = []
        test_data = []

        for bag, score, (sample_id, surgeon_id) in all_embeddings:
            if surgeon_id == test_surgeon:
                test_data.append((bag, score, (sample_id, surgeon_id)))
            else:
                train_data.append((bag, score, (sample_id, surgeon_id)))

        print(f"Train samples: {len(train_data)}")
        print(f"Test samples:  {len(test_data)}")

        # -----------------------------
        # FORMAT FOR HYBRID MIL
        # -----------------------------
        hybrid_train = prepare_hybrid_mil_bags(
            embeddings_list=train_data,
            metrics_df=metrics_df,
            metric_column_name=metric_column_name,
            mean=mean,
            std=std
        )

        hybrid_test = prepare_hybrid_mil_bags(
            embeddings_list=test_data,
            metrics_df=metrics_df,
            metric_column_name=metric_column_name,
            mean=mean,
            std=std
        )

        # -----------------------------
        # TRAIN MODEL
        # -----------------------------
        model, best_rmse = run_mil_training(
            hybrid_train,
            hybrid_test,
            num_epochs=num_epochs,
            lr=lr
        )

        # Store result
        results[test_surgeon]["rmse"] = best_rmse
        fold += 1

    # -----------------------------
    # REPORT SUMMARY
    # -----------------------------
    print("\n======= LOSO SUMMARY =======")
    rmses = []

    for surgeon, metrics in results.items():
        print(f"Surgeon {surgeon} — RMSE: {metrics['rmse']:.4f}")
        rmses.append(metrics['rmse'])

    print("\n------- OVERALL -------")
    print("Mean RMSE:", np.mean(rmses))
    print("Std RMSE: ", np.std(rmses))

    return results


In [13]:
loso_results = run_loso_hybrid_mil(
    all_embeddings=video_tcn_embeddings,
    metrics_df=df_metrics,
    metric_column_name=TPL_METRIC_NAME,
    mean=mean,
    std=std,
    num_epochs=300,
    lr=3e-5
)



======= STARTING LOSO CROSS VALIDATION =======
Total surgeons: 28

FOLD 1/28 — TEST SURGEON: 1
Train samples: 80
Test samples:  3
Starting HYBRID MIL Training on CPU for 300 epochs...
Training Bags: 80 | Validation Bags: 3
Epoch 01 | Train Loss: 2706.7220 | Val Loss (MSE): 2807.9091 | Val RMSE: 52.9897 | Val MAE: 52.5668 | Val Corr: -0.5000
>>> Saved new best HYBRID model checkpoint based on RMSE.
Epoch 02 | Train Loss: 2686.8201 | Val Loss (MSE): 2785.4074 | Val RMSE: 52.7770 | Val MAE: 52.3531 | Val Corr: -0.5000
>>> Saved new best HYBRID model checkpoint based on RMSE.
Epoch 03 | Train Loss: 2667.4528 | Val Loss (MSE): 2763.0631 | Val RMSE: 52.5648 | Val MAE: 52.1400 | Val Corr: -0.5000
>>> Saved new best HYBRID model checkpoint based on RMSE.
Epoch 04 | Train Loss: 2643.7949 | Val Loss (MSE): 2741.7070 | Val RMSE: 52.3613 | Val MAE: 51.9355 | Val Corr: -0.5000
>>> Saved new best HYBRID model checkpoint based on RMSE.
Epoch 05 | Train Loss: 2630.8022 | Val Loss (MSE): 2722.1239 | V

KeyboardInterrupt: 